# Preparação para modelagem preditiva

## 1. Pergunta preditiva

A formulação recomendada é estimar risco de defasagem futura com informações disponíveis antes do desfecho. Exemplos a validar: estar em defasagem, agravar a defasagem ou permanecer em defasagem no ano seguinte.

In [2]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'src').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.features import classificar_colunas

CAMINHO_PROCESSADA = RAIZ / 'data/processed/passos_magicos_clean_eda.csv'
df = pd.read_csv(CAMINHO_PROCESSADA)
df.shape

(2845, 30)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2845 entries, 0 to 2844
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano Nascimento         2845 non-null   int64  
 1   Ano ingresso           2845 non-null   int64  
 2   Avaliador1             2820 non-null   object 
 3   Avaliador2             2820 non-null   object 
 4   Avaliador3             2027 non-null   object 
 5   Avaliador4             1045 non-null   object 
 6   Defasagem              2845 non-null   int64  
 7   Fase                   2845 non-null   object 
 8   Fase Ideal             2845 non-null   object 
 9   Gênero                 2845 non-null   object 
 10  IAA                    2845 non-null   float64
 11  IAN                    2845 non-null   float64
 12  IDA                    2845 non-null   float64
 13  IEG                    2845 non-null   float64
 14  INDE                   2845 non-null   float64
 15  IPS 

## 2. Relação preliminar entre Defasagem e IAN

In [8]:
pd.crosstab(df['Defasagem'], df['IAN'], margins=True)

IAN,2.5,5.0,10.0,All
Defasagem,,,,
-5,1,0,0,1
-4,5,0,0,5
-3,38,0,0,38
-2,0,382,0,382
-1,0,1253,0,1253
0,0,0,1024,1024
1,0,0,120,120
2,0,0,20,20
3,0,0,2,2


O cruzamento deve ser interpretado apenas como validação estrutural. A associação forte indica que IAN e Defasagem não podem ser tratados automaticamente como feature e target independentes no mesmo período. O significado do sinal de Defasagem ainda requer confirmação oficial.

## 3. Classificação preliminar das colunas

In [3]:
catalogo_features = classificar_colunas(df.columns)
catalogo_features.sort_values(['categoria', 'coluna'])

,coluna,categoria,justificativa
9,ano_ingresso,candidata_condicional,Usar somente se disponível antes do desfecho e...
1,fase,candidata_condicional,Usar somente se disponível antes do desfecho e...
8,genero,candidata_condicional,Usar somente se disponível antes do desfecho e...
29,iaa,candidata_condicional,Usar somente se disponível antes do desfecho e...
34,ida,candidata_condicional,Usar somente se disponível antes do desfecho e...
7,idade,candidata_condicional,Usar somente se disponível antes do desfecho e...
30,ieg,candidata_condicional,Usar somente se disponível antes do desfecho e...
37,ing,candidata_condicional,Usar somente se disponível antes do desfecho e...
10,instituicao_de_ensino,candidata_condicional,Usar somente se disponível antes do desfecho e...
32,ipp,candidata_condicional,Usar somente se disponível antes do desfecho e...


## 4. Estratégia temporal recomendada

- Features de 2022 para desfecho de 2023.
- Features de 2023 para desfecho de 2024.
- Janela mais recente reservada para avaliação final.
- RA usado somente para vinculação e separação, nunca como feature.
- Nenhuma coluna medida depois do desfecho pode entrar no treinamento.

## 5. Features candidatas condicionais

Após validação da data de disponibilidade, podem ser avaliadas fase, idade, tempo de programa, indicadores acadêmicos e de engajamento do ano anterior. Variáveis psicossociais exigem justificativa ética, análise por subgrupos e controle de uso.

IAN, Defasagem, Fase Ideal, INDE e Pedra permanecem bloqueados ou condicionais até a definição do target e da janela. Identificadores, nomes, datas de nascimento e avaliadores são proibidos como features.

## 6. Métricas recomendadas

A avaliação futura deverá priorizar recall e F1 da classe de risco, PR-AUC, matriz de confusão e calibração. Também deverá apresentar precisão, prevalência, análise de erros e desempenho por subgrupos. Acurácia isolada não será critério principal.